In [ ]:
!pip install -q pypdf[crypto] anthropic chromadb voyageai gradio

from pypdf import PdfReader
import os

# Caminho do ficheiro (usando o nome exato detetado no sistema)
pdf_path = "/content/Boletim Economico_março2026.pdf"

if not os.path.exists(pdf_path):
    # Tentar procurar ficheiros PDF se o nome exato falhar por causa de caracteres especiais
    pdfs = [f for f in os.listdir('/content/') if f.endswith('.pdf')]
    if pdfs:
        pdf_path = os.path.join('/content/', pdfs[0])
        print(f"→ A usar ficheiro encontrado: {pdf_path}")

if not os.path.exists(pdf_path):
    print(f"❌ Erro: Ficheiro no encontrado em {pdf_path}")
else:
    try:
        reader = PdfReader(pdf_path)
        paginas = []

        for i, page in enumerate(reader.pages, start=1):
            texto = page.extract_text()
            if texto and len(texto.strip()) > 50:
                paginas.append({
                    'pagina': i,
                    'texto': texto.strip()
                })

        print(f"✓ Extradas {len(paginas)} pginas com contedo")
        if paginas:
            print(f"\nPrimeira pgina com texto (excerto):")
            print(paginas[0]['texto'][:500])
    except Exception as e:
        print(f"❌ Erro ao ler o PDF: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 9.1 MB/s eta 0:00:00
✓ Extradas 67 pginas com contedo

Primeira pgina com texto (excerto):
BOLETIM  
ECONÓMICO
Lisboa, 2026  •  www.bportugal.pt 
MAR . 202 6 
 
Em ficheiro anexo são disponibilizados os dados subjacentes  
aos gráficos do Boletim Económico. 
Não são divulgados dados de algumas fontes privadas.


In [ ]:
def criar_chunks(paginas, tamanho_max=1200, overlap=150):
    """
    Cria chunks respeitando parágrafos e mantendo referência à página.
    Overlap garante que conceitos não ficam cortados ao meio.
    """
    chunks = []

    for pag in paginas:
        texto = pag['texto']

        # Dividir por parágrafos (linhas duplas ou pontos finais)
        paragrafos = [p.strip() for p in texto.split('\n\n') if len(p.strip()) > 50]

        chunk_atual = ""
        for par in paragrafos:
            if len(chunk_atual) + len(par) > tamanho_max and chunk_atual:
                chunks.append({
                    'texto': chunk_atual.strip(),
                    'pagina': pag['pagina'],
                    'fonte': f"Boletim Económico BdP, março 2026, p.{pag['pagina']}"
                })
                # Overlap: começar o próximo chunk com últimas palavras
                chunk_atual = chunk_atual[-overlap:] + "\n" + par
            else:
                chunk_atual += "\n" + par

        if chunk_atual.strip():
            chunks.append({
                'texto': chunk_atual.strip(),
                'pagina': pag['pagina'],
                'fonte': f"Boletim Económico BdP, março 2026, p.{pag['pagina']}"
            })

    return chunks

chunks = criar_chunks(paginas)
print(f"✓ {len(chunks)} chunks criados")
print(f"\nExemplo de chunk:")
print(f"Página: {chunks[5]['pagina']}")
print(f"Texto: {chunks[5]['texto'][:300]}...")

✓ 67 chunks criados

Exemplo de chunk:
Página: 10
Texto: 8 
 Banco de Portugal  •  Boletim Económico  •  Março 2026 
Face ao Boletim Económico de dezembro, o crescimento do PIB foi revisto em baixa em 2026 (-0,5 pp) 
e 2027 (-0,1 pp), enquanto a inflação foi revista em alta (0,7 pp e 0,3 pp em 2026 e 2027). O agravamento 
das tensões geopolíticas no Médio...


In [ ]:
import os
from google.colab import userdata
import voyageai
import chromadb
import time

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["VOYAGE_API_KEY"] = userdata.get("VOYAGE_API_KEY")

vo = voyageai.Client()

# Gerar embeddings em batches para evitar rate limit (3 RPM no tier gratuito)
def gerar_embeddings_batch(textos, batch_size=7):
    todos = []
    print(f"⌛ A aguardar 10s antes de iniciar para limpar janela de rate limit...")
    time.sleep(10)

    for i in range(0, len(textos), batch_size):
        batch = textos[i:i+batch_size]
        # Tenta realizar o embedding com retry manual simples se falhar por rate limit
        sucesso = False
        while not sucesso:
            try:
                result = vo.embed(batch, model="voyage-3", input_type="document")
                todos.extend(result.embeddings)
                sucesso = True
            except Exception as e:
                if "rate_limit" in str(e).lower():
                    print("  ⚠️ Rate limit atingido. A aguardar 30s extras...")
                    time.sleep(30)
                else:
                    raise e

        print(f"  Batch {i//batch_size + 1}: {len(todos)}/{len(textos)} embeddings")
        if i + batch_size < len(textos):
            time.sleep(30)  # Pausa obrigatória para respeitar os 3 RPM
    return todos

print("→ A gerar embeddings... (pode demorar 10-15 min com rate limit gratuito)")
textos = [c['texto'] for c in chunks]
embeddings = gerar_embeddings_batch(textos)

# Criar ChromaDB
chroma_client = chromadb.Client()
# Apagar coleção se já existir para evitar erros de duplicado
try:
    chroma_client.delete_collection("bdp_boletim_marco_2026")
except:
    pass

colecao = chroma_client.create_collection("bdp_boletim_marco_2026")

colecao.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    embeddings=embeddings,
    documents=textos,
    metadatas=[{'pagina': c['pagina'], 'fonte': c['fonte']} for c in chunks]
)

print(f"✓ {len(chunks)} chunks indexados no ChromaDB")

→ A gerar embeddings... (pode demorar 10-15 min com rate limit gratuito)
⌛ A aguardar 10s antes de iniciar para limpar janela de rate limit...
  Batch 1: 7/67 embeddings
  Batch 2: 14/67 embeddings
  Batch 3: 21/67 embeddings
  Batch 4: 28/67 embeddings
  Batch 5: 35/67 embeddings
  Batch 6: 42/67 embeddings
  Batch 7: 49/67 embeddings
  Batch 8: 56/67 embeddings
  Batch 9: 63/67 embeddings
  Batch 10: 67/67 embeddings
✓ 67 chunks indexados no ChromaDB


In [ ]:
from anthropic import Anthropic

client = Anthropic()

SYSTEM_PROMPT = """És o EY AI Regulatory Assistant, um assistente avançado de Inteligência Artificial especializado em analisar publicações e previsões económicas do Banco de Portugal (BdP).

O teu público-alvo são consultores e auditores da EY em Financial Services. Eles exigem respostas diretas, rigorosas, baseadas em dados empíricos e formatadas para leitura rápida (scannability).

Vais receber [CONTEXTO_RECUPERADO] contendo excertos de texto e tabelas em Markdown extraídos diretamente dos relatórios do BdP. 

REGRAS DE EXECUÇÃO OBRIGATÓRIAS (ZERO HALLUCINATION):
1. Fidelidade Absoluta: Responde ESTRITAMENTE com base no [CONTEXTO_RECUPERADO]. Não utilizes conhecimento externo, não especules e não inventes dados.
2. Citações Rigorosas: Cada facto, número ou previsão deve ser acompanhado da sua fonte exata no final da frase. Utiliza o formato: [Nome do Boletim, p. X] (ex: [Boletim Económico Março 2026, p. 14]). 
3. Horizontes Temporais: Em projeções financeiras (PIB, inflação, desemprego), especifica sempre o ano a que a métrica se refere e se é um dado histórico ou uma projeção.
4. Integridade de Tabelas e Dados: Se a resposta envolver dados estruturados do contexto, apresenta-os usando tabelas em formato Markdown. Não tentes fazer cálculos matemáticos avançados (ex: taxas de variação não presentes no texto) a menos que explicitamente instruído pelo utilizador.
5. Terminologia Técnica: Preserva o jargão económico e institucional do BdP (ex: "FBCF", "riscos descendentes", "IHPC"). Não simplifiques excessivamente a linguagem.
6. Estrutura Visual: Organiza respostas complexas em secções com títulos (`###`), utiliza *bullet points* para enumerar fatores de risco ou vetores económicos, e coloca os números/percentagens a **negrito** para destaque visual.
7. Tratamento de Lacunas: Se a resposta à pergunta não estiver contida no contexto fornecido, és estritamente obrigado a responder: "A informação recuperada dos Boletins consultados não contém dados para responder a esta questão." Não tentes preencher a lacuna.
8. Tom e Idioma: Mantém um tom altamente profissional, analítico e objetivo. Responde sempre em Português de Portugal (PT-PT).

FECHO DA RESPOSTA:
Termina sempre as tuas respostas substanciais com uma única "Sugestão de Análise Seguinte:" formatada em itálico, propondo uma pergunta relevante que aprofunde o tema com base no contexto disponível, ajudando o consultor da EY a explorar mais o relatório.
"""

def responder(pergunta, k=5):
    """RAG: recupera chunks relevantes e responde com Claude."""
    # 1. Embedding da pergunta
    q_emb = vo.embed([pergunta], model="voyage-3", input_type="query").embeddings[0]

    # 2. Pesquisar chunks mais relevantes
    resultados = colecao.query(query_embeddings=[q_emb], n_results=k)

    # 3. Montar contexto
    contexto = ""
    fontes = []
    for doc, meta in zip(resultados['documents'][0], resultados['metadatas'][0]):
        contexto += f"\n[Fonte: p.{meta['pagina']}]\n{doc}\n---"
        fontes.append(meta['fonte'])

    # 4. Chamar Claude
    resposta = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=1500,
        system=SYSTEM_PROMPT,
        messages=[{
            "role": "user",
            "content": f"""Excertos do Boletim Económico do Banco de Portugal (março 2026):

{contexto}

Pergunta do consultor: {pergunta}"""
        }]
    )

    return {
        'resposta': resposta.content[0].text,
        'fontes': list(set(fontes))
    }

# Testar
r = responder("Quais são as projeções do BdP para o crescimento do PIB português em 2026?")
print(r['resposta'])

## Projeções do BdP para o Crescimento do PIB Português em 2026

De acordo com o Boletim Económico de março de 2026, o Banco de Portugal projeta um **crescimento do PIB de 1,8% em 2026** [p.9].

### Contexto e fatores condicionantes

Este valor representa uma **revisão em baixa** face à projeção de dezembro de 2025, que apontava para um crescimento de 2,3% em 2026 [p.9]. Os principais fatores que explicam esta revisão são:

- **Choque geopolítico:** o ataque lançado pelos Estados Unidos e Israel ao Irão no final de fevereiro, que provocou uma subida abrupta dos preços das matérias-primas energéticas, com impacto negativo na atividade [p.9];
- **Agravamento das condições de financiamento**, que condiciona a atividade ao longo do horizonte de projeção [p.9];
- **Condições meteorológicas extremas** registadas no início do ano [p.9].

### Fatores de suporte ao crescimento em 2026

- **Solidez do mercado de trabalho** (taxa de desemprego projetada em 5,9%) [p.9];
- **Ímpeto associado ao PRR

In [ ]:
import gradio as gr

EXEMPLOS_DEMO = [
    "Quais são as projeções do BdP para o crescimento do PIB em 2026, 2027 e 2028?",
    "Qual o impacto esperado do conflito no Médio Oriente na inflação portuguesa?",
    "Resume os indicadores de acessibilidade à habitação em Portugal",
    "Como evoluiu a literacia financeira em Portugal segundo o BdP?",
    "Qual o impacto dos direitos aduaneiros dos EUA sobre as importações da China para a UE?",
    "Que riscos o BdP identifica para a economia portuguesa em 2026?",
]

def chat_ey(mensagem, historico):
    resultado = responder(mensagem)
    resposta_formatada = resultado['resposta']
    if resultado['fontes']:
        resposta_formatada += f"\n\n---\n**📚 Fontes:** {', '.join(resultado['fontes'][:3])}"
    return resposta_formatada

demo = gr.ChatInterface(
    fn=chat_ey,
    title="🏦 EY Regulatory Assistant — Banco de Portugal",
    description="Assistente conversacional para consultores da EY especializado em publicações do Banco de Portugal. POC: Boletim Económico Março 2026.",
    examples=EXEMPLOS_DEMO,
    theme=gr.themes.Soft(primary_hue="amber"),
)

demo.launch(share=True)  # gera URL público para demo!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://14691a7c0d5e19b648.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
